# 07 · Model Building & Evaluation — FRAUD_ECOMMERCE
**Marker:** `fraud_nb01-07_v2` · **Inputs:** `data/06_*`, `reports/06_run_record.json`, `raw/label_sources/chargebacks.csv`
**Outputs:** `models/v_UTCSTAMP_nbHASH/` (immutable), `models/CANDIDATE.json`, `reports/07_cv_results.csv`, `reports/07_run_record.json`

| Window | Rows | Used for |
|---|---|---|
| **DEV** | train, `payment_ts < 2025-03-18` | CV selection; refit |
| **GAP** | 2025-03-18 → 2025-03-31 | unused (14 days = max audit / alert-review label latency) |
| **VAL** | 2025-04-01 → 2025-06-30 | early stopping, isotonic calibration, operating point, fairness |
| **TEST** | `__split == 'test'` | scored **once**, at the end |

* **CV design.** Five expanding-window folds on DEV. Each validation block is 61 days, with a 14-day gap before it.
* **Label maturity.** In each fold, a training row whose chargeback was resolved after the fold's retrain moment has its label reverted to unadjudicated (0, weight 0.35). This is how the approved "mimic label maturity" default is implemented. The alternative, a flat 143-day embargo equal to the chargeback resolution p95, leaves the first fold with no training data.
* **The final refit uses mature labels.** Masking exists to keep the selection estimate honest, not to weaken the artifact.
* **Selection.** Candidates are chosen on CV PR-AUC (unweighted, hard union label, all fold rows). LightGBM is the pre-registered default, and another family replaces it only if it beats LightGBM by ≥ 5% on CV.
* **Gate (provisional): design-weighted PR-AUC on TEST.**
  * **Why design weights.** `random_audit` samples only the **non-alerted** stratum, and `alert_review` covers the alerted stratum. Each reviewed row is weighted by its stratum's inverse sampling fraction, which estimates population PR-AUC without the incumbent's selection bias.
  * **Absolute floor.** The incumbent's design-weighted PR-AUC on the train period.
  * **Relative requirement.** At least 1.25× the incumbent on the same test sample.
  * **Small-sample rule.** With fewer than 50 audit positives, both halves use the 5th percentile of a stratified paired bootstrap.
  * **Supporting line.** Audit-only PR-AUC is reported but does not gate.
* **No promotion.** This notebook writes a candidate version. `CURRENT.json` is written only after Phase-1 sign-off.

In [ ]:
%pip install -q boto3==1.43.95

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 07_Model_Building_and_Evaluation
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

In [ ]:
# 07.1 Load the encoded stages and restore the trained dtypes from the encoding params
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import loguniform, randint, uniform
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.model_selection import ParameterSampler
import matplotlib.pyplot as plt

P = read_json_s3("data/06_encoding_params.json")
rec06 = read_json_s3("reports/06_run_record.json")
assert rec06["marker"] == MARKER and rec06["feature_hash"] == P["feature_hash"], "06 artifacts are inconsistent"
tree_bytes = read_bytes_s3("data/06_encoded_tree.parquet")
lin_bytes = read_bytes_s3("data/06_encoded_linear.parquet")
DATA_HASH = hashlib.sha256(tree_bytes + lin_bytes).hexdigest()
TREE = pd.read_parquet(io.BytesIO(tree_bytes))
LIN = pd.read_parquet(io.BytesIO(lin_bytes))
del tree_bytes, lin_bytes
META = rec06["meta_columns"]
TREE_FEATURES, LIN_FEATURES = P["tree"]["feature_order"], P["linear"]["feature_order"]


def feature_contract_hash(p):
    core = {"tree": [p["tree"]["feature_order"], p["tree"]["dtypes"], p["tree"]["categorical"]],
            "linear": [p["linear"]["feature_order"], p["linear"]["categorical"]]}
    return hashlib.sha256(json.dumps(core, sort_keys=True).encode()).hexdigest()


def restore_tree_dtypes(X, p):
    """Categories come from the encoder map, never from the data in hand."""
    cols = {c: X[c].astype("float64") for c in p["tree"]["numeric"]}
    for c, levels in p["tree"]["categorical"].items():
        cols[c] = pd.Categorical(X[c].astype(object), categories=levels)
    return pd.DataFrame(cols, index=X.index)[p["tree"]["feature_order"]]


assert feature_contract_hash(P) == P["feature_hash"], "encoding params were edited after 06"
assert TREE["payment_id"].equals(LIN["payment_id"]), "tree and linear stages are not row-aligned"
assert "__split" not in TREE_FEATURES and "__split" not in LIN_FEATURES
meta = TREE[META].reset_index(drop=True)
XT = restore_tree_dtypes(TREE[TREE_FEATURES], P).reset_index(drop=True)
XL = LIN[LIN_FEATURES].astype("float64").reset_index(drop=True)
del TREE, LIN
assert {c: str(XT[c].dtype) for c in XT.columns} == P["tree"]["dtypes"]
X_OF = {"logreg": XL, "lightgbm": XT, "xgboost": XT}
print("tree", XT.shape, "| linear", XL.shape, "| data hash", DATA_HASH[:16])

In [ ]:
# 07.2 Windows, folds and chargeback-maturity masking
GAP_DAYS = 14
VAL_START, VAL_END = pd.Timestamp("2025-04-01"), SPLIT_DATE
DEV_END = VAL_START - pd.Timedelta(days=GAP_DAYS)
FOLD_STARTS = pd.to_datetime(["2024-06-01", "2024-08-01", "2024-10-01", "2024-12-01", "2025-02-01"])
FOLD_BLOCK_DAYS = 61

ts = meta["payment_ts"]
is_train = meta["__split"].eq("train").to_numpy()
DEV = is_train & (ts < DEV_END).to_numpy()
VAL = is_train & ((ts >= VAL_START) & (ts < VAL_END)).to_numpy()
TEST = meta["__split"].eq("test").to_numpy()
AUDIT = meta["label_source"].eq("random_audit").to_numpy()
CB_ROWS = meta["label_source"].eq("chargeback").to_numpy()
LABEL_SOURCE = meta["label_source"].to_numpy()
y_all = meta["is_fraud"].to_numpy()
w_all = meta["sample_weight"].to_numpy()
base_all = meta["baseline_score"].to_numpy()
alert_all = meta["alerted_int"].to_numpy()
assert not (DEV & VAL).any() and not ((DEV | VAL) & TEST).any()
assert (ts[VAL].min() >= VAL_START) and (ts[TEST].min() >= SPLIT_DATE)

cb = pd.read_csv(io.BytesIO(read_bytes_s3("raw/label_sources/chargebacks.csv")), dtype=str)
cb.columns = cb.columns.str.strip()
cb = cb.apply(lambda c: c.str.strip())
won = cb.loc[cb["outcome"].eq("Issuer Won")]
resolved_at = pd.to_datetime(won["resolution_timestamp"], format="%Y-%m-%dT%H:%M:%S", errors="raise")
assert resolved_at.notna().all(), "an 'Issuer Won' chargeback has no resolution timestamp"
resolved_at = resolved_at.groupby(won["payment_id"]).min()
LABEL_KNOWN_AT = meta["payment_id"].map(resolved_at)
_unmatched = int((CB_ROWS & LABEL_KNOWN_AT.isna().to_numpy()).sum())
assert _unmatched == 0, f"{_unmatched} chargeback-labelled rows have no 'Issuer Won' resolution in chargebacks.csv"
known_at = LABEL_KNOWN_AT.to_numpy()

folds, fold_rows = [], []
for i, start in enumerate(FOLD_STARTS):
    end = min(start + pd.Timedelta(days=FOLD_BLOCK_DAYS), DEV_END)
    trm = DEV & (ts < start - pd.Timedelta(days=GAP_DAYS)).to_numpy()
    vam = DEV & ((ts >= start) & (ts < end)).to_numpy()
    masked = trm & CB_ROWS & (known_at > np.datetime64(start))
    y_tr, w_tr = y_all.copy(), w_all.copy()
    y_tr[masked], w_tr[masked] = 0, 0.35
    folds.append({"train": trm, "val": vam, "y_train": y_tr[trm], "w_train": w_tr[trm], "retrain_at": start})
    fold_rows.append({"fold": i, "train_until": (start - pd.Timedelta(days=GAP_DAYS)).date(), "val": f"{start.date()}..{end.date()}",
                      "train_rows": int(trm.sum()), "train_pos_mature": int(y_all[trm].sum()),
                      "masked_chargebacks": int(masked.sum()), "train_pos_used": int(y_tr[trm].sum()),
                      "val_rows": int(vam.sum()), "val_pos": int(y_all[vam].sum()),
                      "val_audit_pos": int(y_all[vam & AUDIT].sum())})
FOLDS = pd.DataFrame(fold_rows)
print(FOLDS.to_string(index=False))
assert (FOLDS.val_pos >= 50).all() and (FOLDS.train_pos_used >= 200).all(), "a fold is too thin to score"
WINDOWS_SUMMARY = {"dev": [str(ts[DEV].min()), str(ts[DEV].max()), int(DEV.sum()), int(y_all[DEV].sum())],
                   "val": [str(ts[VAL].min()), str(ts[VAL].max()), int(VAL.sum()), int(y_all[VAL].sum())],
                   "test": [str(ts[TEST].min()), str(ts[TEST].max()), int(TEST.sum()), int(y_all[TEST].sum())],
                   "gap_days": GAP_DAYS}
print(WINDOWS_SUMMARY)

def design_weights(label_source, alerted, in_scope):
    """Inverse-probability weights for the two-stratum review design inside `in_scope` (numpy bool array).
    Alerted stratum: reviewed through alert_review (near-census). Non-alerted stratum: random_audit sample.
    Rows outside the design (chargeback-only, unadjudicated) get weight 0."""
    label_source, alerted = np.asarray(label_source), np.asarray(alerted)
    assert not ((label_source == "random_audit") & (alerted == 1)).any(), "audit rows must be non-alerted"
    assert not ((label_source == "alert_review") & (alerted == 0)).any(), "alert-review rows must be alerted"
    w = np.zeros(len(label_source), dtype="float64")
    for flag, src in ((1, "alert_review"), (0, "random_audit")):
        stratum = in_scope & (alerted == flag)
        sampled = stratum & (label_source == src)
        assert sampled.sum() > 0, f"no {src} rows in scope"
        w[sampled] = stratum.sum() / sampled.sum()
    return w


def ap_ipw(y, score, w):
    """Design-weighted average precision: an estimate of population PR-AUC from the reviewed sample."""
    s = w > 0
    return float(average_precision_score(y[s], score[s], sample_weight=w[s]))


W_VAL = design_weights(LABEL_SOURCE, alert_all, VAL)
for f in folds:
    f["w_design_val"] = design_weights(LABEL_SOURCE, alert_all, f["val"])[f["val"]]

In [ ]:
# 07.3 Candidates, search spaces, selection rule (all fixed before any result is seen)
SEARCH_ITER = 20
DEFAULT_FAMILY = "lightgbm"
HYSTERESIS = 0.05
N_JOBS = -1          # threaded boosters: results are reproducible here but not guaranteed bit-identical across machines
FAMILIES = ["logreg", "lightgbm", "xgboost"]
SPACES = {
    "logreg": {"C": loguniform(1e-3, 10)},
    "lightgbm": {"n_estimators": randint(200, 801), "learning_rate": loguniform(0.02, 0.2),
                 "num_leaves": randint(15, 128), "min_child_samples": randint(20, 401),
                 "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.5, 0.5),
                 "reg_lambda": loguniform(1e-3, 10)},
    "xgboost": {"n_estimators": randint(200, 801), "learning_rate": loguniform(0.02, 0.2),
                "max_depth": randint(3, 11), "min_child_weight": loguniform(1, 50),
                "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.5, 0.5),
                "reg_lambda": loguniform(1e-3, 10)},
}


def make_model(family, params):
    if family == "logreg":
        return LogisticRegression(penalty="l2", solver="lbfgs", max_iter=3000, **params)
    if family == "lightgbm":
        return lgb.LGBMClassifier(objective="binary", subsample_freq=1, random_state=SEED, n_jobs=N_JOBS,
                                  deterministic=True, force_row_wise=True, verbose=-1, **params)
    if family == "xgboost":
        return xgb.XGBClassifier(objective="binary:logistic", tree_method="hist", enable_categorical=True,
                                 max_cat_to_onehot=1, random_state=SEED, n_jobs=N_JOBS, **params)
    raise ValueError(family)


def ap_or_nan(y, p):
    return float(average_precision_score(y, p)) if y.sum() > 0 else float("nan")


SEARCH_CONFIG = {"search_iter": SEARCH_ITER, "seed": SEED, "spaces": {f: {k: repr(v.dist.name) + repr(v.args)
                 for k, v in s.items()} for f, s in SPACES.items()}, "folds": FOLDS.astype(str).to_dict("records"),
                 "feature_hash": P["feature_hash"], "data_hash": DATA_HASH}
CONFIG_HASH = hashlib.sha256(json.dumps(SEARCH_CONFIG, sort_keys=True).encode()).hexdigest()[:16]
print("search config hash:", CONFIG_HASH)

In [ ]:
# 07.4 Incumbent context and the gate — fixed here, before the test set is touched
W_TRAIN = design_weights(LABEL_SOURCE, alert_all, is_train)
FLOOR = ap_ipw(y_all, base_all, W_TRAIN)
n_aud_tr = int((is_train & AUDIT).sum())
GATE = {
    "metric": "design-weighted PR-AUC (average precision), hard union label",
    "population": ("TEST reviewed sample: alert_review rows (alerted stratum) + random_audit rows (non-alerted stratum), "
                   "each weighted by its stratum's inverse sampling fraction"),
    "why_not_audit_only": "random_audit contains no alerted rows, so audit-only compares models only where the incumbent declined to act",
    "absolute_floor": FLOOR,
    "absolute_floor_derivation": (f"incumbent payment_risk_score design-weighted PR-AUC on the train period "
                                  f"(audit rows {n_aud_tr}, audit positives {int(y_all[is_train & AUDIT].sum())}); "
                                  "fixed before test scoring; note train prevalence differs from test"),
    "relative_ratio": 1.25,
    "relative_derivation": "model PR-AUC must be >= 1.25x the incumbent's on the same weighted test sample (approved default)",
    "small_sample_rule": ("if TEST audit positives < 50: both halves use the 5th percentile of a stratified paired "
                          "bootstrap (B=1000)"),
    "provisional": True,
    "review_trigger": "cost figures received -> derive operating threshold and floor from the cost matrix",
}
print(json.dumps(GATE, indent=1))
FOLD_CONTEXT = FOLDS[["fold", "val"]].assign(
    incumbent_ap_val_all_rows=[ap_or_nan(y_all[f["val"]], base_all[f["val"]]) for f in folds])
print(FOLD_CONTEXT.round(4).to_string(index=False),
      "\n(all-row incumbent AP is inflated by alert-review selection; context only)")

In [ ]:
# 07.5 Cross-validated random search (custom loop: labels differ per fold because of maturity masking)
RESUME = True        # reuse finished families from reports/07_cv_results.csv when the config hash matches
prior = None
if RESUME and key_exists("reports/07_cv_results.csv"):
    prior = pd.read_csv(io.BytesIO(read_bytes_s3("reports/07_cv_results.csv")))
    prior = prior[prior["config_hash"].eq(CONFIG_HASH)]
    print("resumable rows:", len(prior))


def cv_evaluate(family, params):
    X = X_OF[family]
    s_all, s_ipw = [], []
    for f in folds:
        m = make_model(family, params)
        m.fit(X[f["train"]], f["y_train"], sample_weight=f["w_train"])
        p = m.predict_proba(X[f["val"]])[:, 1]
        yv = y_all[f["val"]]
        s_all.append(ap_or_nan(yv, p))
        s_ipw.append(ap_ipw(yv, p, f["w_design_val"]))
    return s_all, s_ipw


CV_ROWS = []
for family in FAMILIES:
    done = prior[prior["family"].eq(family)] if prior is not None else None
    if done is not None and len(done) == SEARCH_ITER:
        CV_ROWS += done.to_dict("records")
        print(f"{family}: reused {len(done)} rows")
        continue
    t0 = time.time()
    for i, params in enumerate(ParameterSampler(SPACES[family], n_iter=SEARCH_ITER, random_state=SEED)):
        params = {k: (v.item() if hasattr(v, "item") else v) for k, v in params.items()}
        s_all, s_ipw = cv_evaluate(family, params)
        CV_ROWS.append({"config_hash": CONFIG_HASH, "family": family, "iter": i,
                        "params": json.dumps(params, sort_keys=True),
                        "cv_ap_mean": float(np.mean(s_all)), "cv_ap_std": float(np.std(s_all, ddof=1)),
                        **{f"fold{j}_ap": v for j, v in enumerate(s_all)},
                        "cv_ipw_ap_mean": float(np.mean(s_ipw)), "cv_ipw_ap_std": float(np.std(s_ipw, ddof=1))})
        print(f"  {family} {i + 1}/{SEARCH_ITER}  cv_ap={CV_ROWS[-1]['cv_ap_mean']:.4f}  ({time.time() - t0:.0f}s)")
    put_bytes_s3(pd.DataFrame(CV_ROWS).to_csv(index=False).encode(), "reports/07_cv_results.csv")
CV = pd.DataFrame(CV_ROWS)

In [ ]:
# 07.6 Selection on CV only, with family-switch hysteresis
best = CV.sort_values("cv_ap_mean", ascending=False).groupby("family", sort=False).head(1).set_index("family")
cv_table = best[["cv_ap_mean", "cv_ap_std", "cv_ipw_ap_mean", "cv_ipw_ap_std", "params"]].sort_values(
    "cv_ap_mean", ascending=False)
print(cv_table.to_string())
leader = cv_table.index[0]
default_score = cv_table.loc[DEFAULT_FAMILY, "cv_ap_mean"]
margin = cv_table.loc[leader, "cv_ap_mean"] / default_score - 1
if leader != DEFAULT_FAMILY and margin >= HYSTERESIS:
    SELECTED = leader
    SELECTION_REASON = f"{leader} beats default {DEFAULT_FAMILY} by {margin:.1%} (>= {HYSTERESIS:.0%}) on CV PR-AUC"
else:
    SELECTED = DEFAULT_FAMILY
    SELECTION_REASON = (f"default {DEFAULT_FAMILY} retained; CV leader {leader} margin {margin:.1%} < {HYSTERESIS:.0%}"
                        if leader != DEFAULT_FAMILY else f"{DEFAULT_FAMILY} is the CV leader")
BEST_PARAMS = {fam: json.loads(cv_table.loc[fam, "params"]) for fam in FAMILIES}
print("SELECTED:", SELECTED, "|", SELECTION_REASON)

In [ ]:
# 07.7 Refit every family's best config on DEV (mature labels); early stopping on VAL; bind the selection once
MAX_TREES, ES_ROUNDS = 3000, 100
y_dev, w_dev, y_val = y_all[DEV], w_all[DEV], y_all[VAL]
FITTED, RESULTS = {}, {}
for fam in FAMILIES:
    X, params = X_OF[fam], BEST_PARAMS[fam]
    t0 = time.time()
    if fam == "lightgbm":
        m = make_model(fam, {**params, "n_estimators": MAX_TREES})
        m.set_params(metric="average_precision")
        m.fit(X[DEV], y_dev, sample_weight=w_dev, eval_set=[(X[VAL], y_val)],
              callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)])
        n_trees = int(m.best_iteration_)
    elif fam == "xgboost":
        m = make_model(fam, {**params, "n_estimators": MAX_TREES})
        m.set_params(eval_metric="aucpr", early_stopping_rounds=ES_ROUNDS)
        m.fit(X[DEV], y_dev, sample_weight=w_dev, eval_set=[(X[VAL], y_val)], verbose=False)
        n_trees = int(m.best_iteration) + 1
    else:
        m = make_model(fam, params)
        m.fit(X[DEV], y_dev, sample_weight=w_dev)
        n_trees = None
        assert m.n_iter_[0] < m.max_iter, "logistic regression did not converge"
    assert n_trees is None or n_trees < MAX_TREES, f"{fam}: early stopping never triggered"
    p_val = m.predict_proba(X[VAL])[:, 1]
    FITTED[fam] = {"model": m, "params": params, "n_trees": n_trees}
    RESULTS[fam] = {"cv_ap_mean": float(cv_table.loc[fam, "cv_ap_mean"]), "val_ap": float(average_precision_score(y_val, p_val)),
                    "val_roc_auc": float(roc_auc_score(y_val, p_val)),
                    "val_ipw_ap": ap_ipw(y_val, p_val, W_VAL[VAL]),
                    "val_audit_ap": ap_or_nan(y_val[AUDIT[VAL]], p_val[AUDIT[VAL]]),
                    "n_trees": n_trees, "fit_seconds": round(time.time() - t0, 1)}
RESULTS["incumbent"] = {"val_ap": float(average_precision_score(y_val, base_all[VAL])),
                        "val_roc_auc": float(roc_auc_score(y_val, base_all[VAL])),
                        "val_ipw_ap": ap_ipw(y_val, base_all[VAL], W_VAL[VAL]),
                        "val_audit_ap": ap_or_nan(y_val[AUDIT[VAL]], base_all[VAL][AUDIT[VAL]])}
print(pd.DataFrame(RESULTS).T.round(4).to_string())

best_model = FITTED[SELECTED]["model"]
best_params = FITTED[SELECTED]["params"]
X_SEL = X_OF[SELECTED]
EXPECTED_MODULE = {"logreg": "sklearn", "lightgbm": "lightgbm", "xgboost": "xgboost"}[SELECTED]
assert type(best_model).__module__.split(".")[0] == EXPECTED_MODULE
assert float(average_precision_score(y_val, best_model.predict_proba(X_SEL[VAL])[:, 1])) == RESULTS[SELECTED]["val_ap"]
if RESULTS[SELECTED]["val_ap"] > 0.9:
    print("FLAG: validation PR-AUC > 0.9 — hunt for leakage before trusting this")

In [ ]:
# 07.8 Isotonic calibration on VAL, reliability, and the provisional operating point
raw_val = best_model.predict_proba(X_SEL[VAL])[:, 1]
iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(raw_val, y_val)
cal_val = iso.predict(raw_val)


def reliability(y, p, bins=10):
    q = pd.qcut(pd.Series(p).rank(method="first"), bins, labels=False)
    t = pd.DataFrame({"bin": q, "p": p, "y": y}).groupby("bin").agg(n=("y", "size"), mean_pred=("p", "mean"),
                                                                    observed=("y", "mean"))
    ece = float((t.n / t.n.sum() * (t.mean_pred - t.observed).abs()).sum())
    return t, ece


rel_raw, ece_raw = reliability(y_val, raw_val)
rel_cal, ece_cal = reliability(y_val, cal_val)
CALIBRATION = {"method": "isotonic on VAL (hard label, unweighted)",
               "target": "observed union-label rate, not the true fraud rate (unadjudicated rows count as 0)",
               "val_brier_raw": float(brier_score_loss(y_val, raw_val)), "val_brier_cal": float(brier_score_loss(y_val, cal_val)),
               "val_ece_raw": ece_raw, "val_ece_cal": ece_cal,
               "note": "VAL metrics are in-sample for the calibrator (and VAL also set early stopping); TEST is the honest read"}
print(json.dumps(CALIBRATION, indent=1))
print(rel_cal.round(5).to_string())

# Operating point: cost figures are pending, so match the incumbent's alert volume on VAL (reviewer capacity)
alert_rate_val = float(alert_all[VAL].mean())
THRESHOLD_RAW = float(np.quantile(raw_val, 1 - alert_rate_val))


def op_metrics(y, flag):
    tp = int((flag & (y == 1)).sum())
    return {"alert_rate": float(flag.mean()), "precision": tp / max(int(flag.sum()), 1),
            "recall": tp / max(int(y.sum()), 1), "alerts": int(flag.sum())}


OPERATING = {"rule": "top-k by raw score, k = incumbent alert rate on VAL (provisional until cost figures)",
             "incumbent_alert_rate_val": alert_rate_val, "threshold_raw": THRESHOLD_RAW,
             "threshold_calibrated": float(iso.predict([THRESHOLD_RAW])[0]),
             "val_model": op_metrics(y_val, raw_val >= THRESHOLD_RAW),
             "val_incumbent": op_metrics(y_val, alert_all[VAL] == 1)}
print(json.dumps(OPERATING, indent=1))

In [ ]:
# 07.9 Explanations: native TreeSHAP (verified to sum to the margin) or linear contributions
rng = np.random.default_rng(SEED)
take = np.sort(rng.choice(np.flatnonzero(VAL), size=min(5000, int(VAL.sum())), replace=False))
Xs = X_SEL.iloc[take]
if SELECTED == "lightgbm":
    contrib = best_model.booster_.predict(Xs, pred_contrib=True, num_iteration=best_model.best_iteration_)
    margin = best_model.booster_.predict(Xs, raw_score=True, num_iteration=best_model.best_iteration_)
elif SELECTED == "xgboost":
    dm = xgb.DMatrix(Xs, enable_categorical=True)
    rng_it = (0, best_model.best_iteration + 1)
    contrib = best_model.get_booster().predict(dm, pred_contribs=True, iteration_range=rng_it)
    margin = best_model.get_booster().predict(dm, output_margin=True, iteration_range=rng_it)
else:
    contrib = np.column_stack([Xs.to_numpy() * best_model.coef_[0], np.full(len(Xs), best_model.intercept_[0])])
    margin = best_model.decision_function(Xs)
assert np.allclose(contrib.sum(axis=1), margin, atol=1e-4), "explanations do not add up to the model margin"
IMPORTANCE = pd.Series(np.abs(contrib[:, :-1]).mean(axis=0), index=Xs.columns).sort_values(ascending=False)
top_share = float(IMPORTANCE.iloc[0] / IMPORTANCE.sum())
print(IMPORTANCE.head(25).round(4).to_string())
if top_share > 0.5:
    print(f"FLAG: {IMPORTANCE.index[0]} carries {top_share:.0%} of total attribution — inspect for leakage")
ax = IMPORTANCE.head(25)[::-1].plot.barh(figsize=(7, 8), title=f"{SELECTED}: mean |contribution| (VAL sample)")
ax.set_xlabel("mean |contribution| (log-odds)")
plt.tight_layout()
plt.show()

In [ ]:
# 07.10 Geographic disparity (no direct protected attributes exist; city / city_tier are the flagged proxies)
GROUPS = pd.DataFrame({"city": meta["city"].astype(str),
                       "city_tier": XT["city_tier"].astype("int64").astype(str)})


def group_table(mask, raw, cal, group_col):
    d = pd.DataFrame({"g": GROUPS.loc[mask, group_col].to_numpy(), "y": y_all[mask],
                      "alert": (raw >= THRESHOLD_RAW).astype(int), "cal": cal})
    t = d.groupby("g").agg(n=("y", "size"), positives=("y", "sum"), alert_rate=("alert", "mean"),
                           mean_cal=("cal", "mean"), observed_rate=("y", "mean"))
    t["recall"] = d[d.y.eq(1)].groupby("g")["alert"].mean()
    t["selection_rate_ratio"] = t.alert_rate / t.alert_rate.max()
    t["equal_opportunity_diff"] = t.recall - t.recall.max()
    t["calibration_gap"] = t.mean_cal - t.observed_rate
    return t


FAIRNESS = {}
for col in ["city_tier", "city"]:
    t = group_table(VAL, raw_val, cal_val, col)
    print(f"\n{col} (VAL)\n", t.round(4).to_string())
    FAIRNESS[col] = {"val": t.round(5).reset_index().to_dict("records"),
                     "min_selection_rate_ratio": float(t.selection_rate_ratio.min()),
                     "four_fifths_flag": bool(t.selection_rate_ratio.min() < 0.8)}
FAIRNESS["note"] = ("no protected attributes in the data, so recoverability cannot be measured; geographic disparity "
                    "is reported per the notebook-01 decision; four-fifths is a screening flag, not a pass condition")

In [ ]:
# 07.11 TEST — scored once. Every candidate is reported; only the CV-selected model is gated.
assert "TEST_SCORED" not in globals(), "the test set was already scored in this kernel — restart and run all"
TEST_SCORED = True
y_t = y_all[TEST]
aud_t = AUDIT[TEST]
base_t = base_all[TEST]
test_raw = {fam: FITTED[fam]["model"].predict_proba(X_OF[fam][TEST])[:, 1] for fam in FAMILIES}
raw_t = test_raw[SELECTED]
cal_t = iso.predict(raw_t)

W_TEST = design_weights(LABEL_SOURCE, alert_all, TEST)
REPORT = {}
for name, scores in [*test_raw.items(), ("incumbent", base_t)]:
    REPORT[name] = {"test_ap_ipw": ap_ipw(y_t, scores, W_TEST[TEST]),
                    "test_ap_all": float(average_precision_score(y_t, scores)),
                    "test_ap_audit_only": ap_or_nan(y_t[aud_t], scores[aud_t])}
print(pd.DataFrame(REPORT).T.round(4).to_string())

model_full = np.full(len(y_all), np.nan)
model_full[TEST] = raw_t
idx_a = np.flatnonzero(TEST & (W_TEST > 0) & (alert_all == 1))
idx_n = np.flatnonzero(TEST & (W_TEST > 0) & (alert_all == 0))
N_POS = int(y_all[idx_n].sum())
ap_model = REPORT[SELECTED]["test_ap_ipw"]
ap_base = REPORT["incumbent"]["test_ap_ipw"]
B = 1000
brng = np.random.default_rng(SEED)
boots = np.empty((B, 3))
for b_i in range(B):
    s = np.concatenate([brng.choice(idx_a, len(idx_a)), brng.choice(idx_n, len(idx_n))])
    am = average_precision_score(y_all[s], model_full[s], sample_weight=W_TEST[s])
    ab = average_precision_score(y_all[s], base_all[s], sample_weight=W_TEST[s])
    boots[b_i] = (am, ab, am / ab)
assert np.isfinite(boots).all()
SMALL = N_POS < 50
abs_value = float(np.percentile(boots[:, 0], 5)) if SMALL else ap_model
rel_value = float(np.percentile(boots[:, 2], 5)) if SMALL else ap_model / ap_base
GATE_RESULT = {
    "sample_rows_alerted_stratum": int(len(idx_a)), "sample_rows_audit_stratum": int(len(idx_n)),
    "positives_alerted_stratum": int(y_all[idx_a].sum()), "audit_positives": N_POS,
    "stratum_weights": {"alerted": float(W_TEST[idx_a][0]), "non_alerted": float(W_TEST[idx_n][0])},
    "ipw_prevalence_test": float((W_TEST * y_all).sum() / W_TEST.sum()),
    "small_sample_rule_applied": SMALL,
    "model_ap": ap_model, "incumbent_ap": ap_base, "ratio_point": ap_model / ap_base,
    "model_ap_ci90": [float(np.percentile(boots[:, 0], 5)), float(np.percentile(boots[:, 0], 95))],
    "ratio_ci90": [float(np.percentile(boots[:, 2], 5)), float(np.percentile(boots[:, 2], 95))],
    "absolute": {"value": abs_value, "floor": FLOOR, "passed": bool(abs_value >= FLOOR)},
    "relative": {"value": rel_value, "required": GATE["relative_ratio"], "passed": bool(rel_value >= GATE["relative_ratio"])},
    "bootstrap": {"B": B, "seed": SEED, "paired": True, "stratified": True},
    "audit_only_supporting": {"model": REPORT[SELECTED]["test_ap_audit_only"],
                              "incumbent": REPORT["incumbent"]["test_ap_audit_only"]},
}
GATE_RESULT["passed"] = GATE_RESULT["absolute"]["passed"] and GATE_RESULT["relative"]["passed"]
print(json.dumps(GATE_RESULT, indent=1))

rel_t, ece_t = reliability(y_t, cal_t)
cb_test = cb[cb["payment_id"].isin(set(meta.loc[TEST, "payment_id"]))]
adj_t = meta["label_source"].ne("unadjudicated").to_numpy()[TEST]
TEST_METRICS = {
    "ap_all": REPORT[SELECTED]["test_ap_all"], "roc_auc_all": float(roc_auc_score(y_t, raw_t)),
    "ap_adjudicated_only": float(average_precision_score(y_t[adj_t], raw_t[adj_t])),
    "brier_cal": float(brier_score_loss(y_t, cal_t)), "ece_cal": ece_t,
    "mean_cal_vs_observed": [float(cal_t.mean()), float(y_t.mean())],
    "operating_model": op_metrics(y_t, raw_t >= THRESHOLD_RAW),
    "operating_incumbent": op_metrics(y_t, alert_all[TEST] == 1),
    "label_maturity": {"test_chargebacks": int(len(cb_test)),
                       "pending": int(cb_test["outcome"].eq("Pending").sum()),
                       "note": "test positives are undercounted where they depend on unresolved chargebacks"},
}
print(json.dumps(TEST_METRICS, indent=1, default=_jsonable))
t_fair = group_table(TEST, raw_t, cal_t, "city_tier")
print("\ncity_tier (TEST)\n", t_fair.round(4).to_string())
FAIRNESS["city_tier"]["test"] = t_fair.round(5).reset_index().to_dict("records")

In [ ]:
# 07.12 Persist an immutable candidate version (no promotion)
created = datetime.now(timezone.utc)
fit_signature = {"family": SELECTED, "params": best_params, "n_trees": FITTED[SELECTED]["n_trees"],
                 "feature_hash": P["feature_hash"], "data_hash": DATA_HASH, "config_hash": CONFIG_HASH}
sig = hashlib.sha256(json.dumps(fit_signature, sort_keys=True).encode()).hexdigest()
VERSION = f"v_{created:%Y%m%dT%H%M%SZ}_nb{sig[:7]}"
PREFIX = f"models/{VERSION}/"
assert s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("KeyCount", 0) == 0, f"{PREFIX} exists — versions are immutable"

# the object about to be written is the object that was selected and reported
assert FITTED[SELECTED]["model"] is best_model
assert type(best_model).__module__.split(".")[0] == EXPECTED_MODULE
assert float(average_precision_score(y_val, best_model.predict_proba(X_SEL[VAL])[:, 1])) == RESULTS[SELECTED]["val_ap"]

input_family = "linear" if SELECTED == "logreg" else "tree"
manifest = {
    "version": VERSION, "created_utc": created.isoformat(timespec="seconds"), "marker": MARKER,
    "family": SELECTED, "estimator_class": f"{type(best_model).__module__}.{type(best_model).__name__}",
    "params": best_params, "n_trees": FITTED[SELECTED]["n_trees"], "selection_reason": SELECTION_REASON,
    "input_family": input_family,
    "feature_order": P[input_family]["feature_order"],
    "feature_dtypes": P["tree"]["dtypes"] if input_family == "tree" else {c: "float64" for c in P["linear"]["feature_order"]},
    "encoder_maps": P[input_family]["categorical"],
    "feature_hash": P["feature_hash"], "preprocessor": "preprocessor.json (06 encoding params, fitted on train)",
    "contract_version": CONTRACT_VERSION, "data_hash_06": DATA_HASH, "search_config_hash": CONFIG_HASH,
    "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT,
    "git_sha": None, "git_note": "run in Colab, not from a git checkout — commit notebooks 01-07 and record the SHA at sign-off",
    "seed": SEED, "determinism": "LightGBM deterministic=True; n_jobs=-1 threaded boosters are not guaranteed bit-identical across machines",
    "windows": WINDOWS_SUMMARY, "folds": FOLDS.astype(str).to_dict("records"),
    "label_weights": {"alert_review": 1.0, "random_audit": 1.0, "chargeback": 0.6, "unadjudicated": 0.35},
    "calibrator": "calibrator.pkl (isotonic, fitted on VAL)", "operating_point": OPERATING,
}
metrics = {
    "version": VERSION, "training_date": created.date().isoformat(), "gate": {**GATE, "result": GATE_RESULT},
    "gate_passed": GATE_RESULT["passed"], "provisional": True, "review_trigger": GATE["review_trigger"],
    "cv": cv_table.reset_index().to_dict("records"), "validation": RESULTS, "calibration": CALIBRATION,
    "test_report_all_candidates": REPORT, "test": TEST_METRICS, "fairness": FAIRNESS,
    "top_features": IMPORTANCE.head(25).round(6).to_dict(), "top_feature_share": top_share,
    "flags": {"val_ap_above_0.9": RESULTS[SELECTED]["val_ap"] > 0.9, "single_feature_dominance": top_share > 0.5,
              "four_fifths_city_tier": FAIRNESS["city_tier"]["four_fifths_flag"],
              "four_fifths_city": FAIRNESS["city"]["four_fifths_flag"]},
}

g = GATE_RESULT
card = f"""# Model card — FRAUD_ECOMMERCE {VERSION}

**Status:** candidate, not promoted. Gate {'PASSED' if g['passed'] else 'NOT PASSED'} (**provisional**, pending cost figures).

## Purpose
This model scores e-commerce payments for fraud risk and orders them for review. It is compared against the incumbent `payment_risk_score`.

## Data and label
* **Source.** `s3://{BUCKET}/data/06_encoded_{input_family}.parquet` (data hash `{DATA_HASH[:16]}`), contract `{CONTRACT_VERSION}`.
* **Label.** The union of confirmed alerts, audit fraud, and Issuer-Won chargebacks.
* **Weights.** Alert review and audit rows 1.0, chargeback rows 0.60, unadjudicated rows 0.35.
* **Split.** Chronological at {SPLIT_DATE.date()}, stamped as `__split`.
* **Windows.** DEV `{WINDOWS_SUMMARY['dev'][0][:10]}..{WINDOWS_SUMMARY['dev'][1][:10]}`, then a {GAP_DAYS}-day gap, then VAL `{WINDOWS_SUMMARY['val'][0][:10]}..{WINDOWS_SUMMARY['val'][1][:10]}`, then TEST.

## Model
* **Family.** `{SELECTED}` ({SELECTION_REASON}).
* **Parameters.** `{json.dumps(best_params)}`, trees: {FITTED[SELECTED]['n_trees']}.
* **Features.** {len(P[input_family]['feature_order'])} {input_family} features, fixed by name (see `reports/06_run_record.json` for every exclusion and its reason).
* **Calibration.** Isotonic, fitted on VAL, targeting the observed union-label rate.

## Performance
* **CV PR-AUC** (all fold rows): {RESULTS[SELECTED]['cv_ap_mean']:.4f}.
* **VAL PR-AUC:** all rows {RESULTS[SELECTED]['val_ap']:.4f} vs incumbent {RESULTS['incumbent']['val_ap']:.4f}; design-weighted {RESULTS[SELECTED]['val_ipw_ap']:.4f} vs incumbent {RESULTS['incumbent']['val_ipw_ap']:.4f}.
* **TEST, design-weighted.** Alerted stratum {g['sample_rows_alerted_stratum']} rows (positives {g['positives_alerted_stratum']}); audit stratum {g['sample_rows_audit_stratum']} rows (positives {g['audit_positives']}).
  * PR-AUC: model {g['model_ap']:.4f} (90% CI {g['model_ap_ci90'][0]:.4f}–{g['model_ap_ci90'][1]:.4f}) vs incumbent {g['incumbent_ap']:.4f}.
  * Ratio: {g['ratio_point']:.2f} (90% CI {g['ratio_ci90'][0]:.2f}–{g['ratio_ci90'][1]:.2f}).
  * Audit-only PR-AUC (supporting): model {g['audit_only_supporting']['model']:.4f} vs incumbent {g['audit_only_supporting']['incumbent']:.4f}.
* **Gate.**
  * Absolute: {g['absolute']['value']:.4f} ≥ floor {FLOOR:.4f} → {g['absolute']['passed']}.
  * Relative: {g['relative']['value']:.2f} ≥ {GATE['relative_ratio']} → {g['relative']['passed']}.
  * Small-sample rule applied: {g['small_sample_rule_applied']}.
* **Operating point** (provisional; matched to the incumbent's alert volume):
  * Model on TEST: precision {TEST_METRICS['operating_model']['precision']:.3f}, recall {TEST_METRICS['operating_model']['recall']:.3f}.
  * Incumbent on TEST: precision {TEST_METRICS['operating_incumbent']['precision']:.3f}, recall {TEST_METRICS['operating_incumbent']['recall']:.3f}.

## Fairness and personal data
* **Regime.** India DPDP Act 2023. Lawful basis: fraud prevention. No direct protected attributes are present in the data.
* **Geographic disparity.** Minimum selection-rate ratio on VAL is {FAIRNESS['city_tier']['min_selection_rate_ratio']:.2f} by city_tier and {FAIRNESS['city']['min_selection_rate_ratio']:.2f} by city.
* **Pincodes, IPs and identifiers** are excluded as inputs.

## Known limitations
* **Immature test labels.** {TEST_METRICS['label_maturity']['pending']} of {TEST_METRICS['label_maturity']['test_chargebacks']} test chargebacks are still pending, so full-test PR-AUC is understated.
* **Biased all-row labels.** Alert-review labels exist only where the incumbent fired, so all-row metrics favour the incumbent.
* **Audit covers only non-alerted payments.** The gate therefore weights the two review strata by their inverse sampling fractions.
* **Small audit sample.** The test audit stratum has {g['audit_positives']} positives, so confidence intervals are wide.
* **Velocity features need online state at serving time.** Phase 2 has to decide how the per-key first-seen state is served.
* **Snapshot attributes** (`prior_orders_12m`, `prior_return_rate`, merchant fields) are not point-in-time. They are accepted per the contract.
* **Retry rows** keep their artefactual 0 labels, per the locked decision.
* **Shared validation window.** Early stopping, calibration and the operating point all use VAL. TEST is the independent read.
* **No git SHA** is recorded, because the notebooks run in Colab.
"""

ref = XT.loc[is_train].copy()
ref["score_raw"] = best_model.predict_proba(X_SEL[is_train])[:, 1]
ref["score_cal"] = iso.predict(ref["score_raw"].to_numpy())
assert "__split" not in ref.columns

save_model_s3(best_model, PREFIX + "model.pkl")
save_model_s3(iso, PREFIX + "calibrator.pkl")
save_json_s3(P, PREFIX + "preprocessor.json")
save_json_s3(P[input_family]["feature_order"], PREFIX + "feature_names.json")
save_s3(ref, PREFIX + "reference.parquet")
save_json_s3(metrics, PREFIX + "metrics.json")
put_bytes_s3(card.encode(), PREFIX + "model_card.md")
save_json_s3(manifest, PREFIX + "manifest.json")       # written last: its presence marks a complete version
reloaded = load_model_s3(PREFIX + "model.pkl")
assert np.array_equal(reloaded.predict_proba(X_SEL[VAL])[:, 1], best_model.predict_proba(X_SEL[VAL])[:, 1]), \
    "reloaded artifact predicts differently"
save_json_s3({"version": VERSION, "prefix": PREFIX, "created_utc": manifest["created_utc"],
              "gate_passed": GATE_RESULT["passed"], "provisional": True,
              "note": "candidate only — models/CURRENT.json is written after Phase-1 sign-off"}, "models/CANDIDATE.json")
save_json_s3({"notebook": "07_Model_Building_and_Evaluation", "marker": MARKER, "version": VERSION,
              "selected": SELECTED, "selection_reason": SELECTION_REASON, "gate_passed": GATE_RESULT["passed"],
              "provisional": True, "gate": GATE_RESULT, "validation": RESULTS, "test_report": REPORT,
              "flags": metrics["flags"], "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT},
             "reports/07_run_record.json")
print("\nVERSION:", VERSION, "| gate passed:", GATE_RESULT["passed"], "(provisional)")